# 🎬 Reels — Free-GPU Talking Avatar (MuseTalk)

Generate a talking-avatar reel from a **script + one photo**, 100% free on a Colab **T4 GPU**.

**Runtime → Change runtime type → GPU (T4)** before running.

Pipeline: `script → edge-tts voice → MuseTalk lip-sync → 9:16 reel`.

> ⚠️ **Run the cells top-to-bottom, one at a time — do NOT use 'Run all'.**
> Cell 2 installs Python 3.10 via condacolab and **restarts the kernel** (needed so MuseTalk's
> mmcv/mmpose wheels install). After it restarts, continue from cell 3 downward.

In [ ]:
# 1) Check the GPU (must show a Tesla T4 / similar)
!nvidia-smi -L
import sys; print('python', sys.version.split()[0])

In [ ]:
# 2) Install Python 3.10 (condacolab) so MuseTalk's mmcv/mmpose have prebuilt wheels.
#    ⚠️ This RESTARTS the kernel — that is expected. After it restarts, run cell 3 onward.
!pip install -q condacolab
import condacolab
condacolab.install()  # kernel restarts here

In [ ]:
# 3) (after restart) confirm we are on Python 3.10 + conda
import sys; print('python', sys.version.split()[0])
import condacolab; condacolab.check()

In [ ]:
# 4) Get this repo (scripts: tts.py, reel_utils.py, avatars/, examples/)
import os
if not os.path.isdir('/content/reels'):
    !git clone https://github.com/amsinghnavdeep/reels.git /content/reels
%cd /content/reels
!git pull -q
!pip -q install edge-tts pydub pyyaml

In [ ]:
# 5) Install MuseTalk + mmlab (one-time, ~8-10 min).
#    torch 2.1.0+cu118 has matching PREBUILT mmcv/mmdet/mmpose wheels (no slow source build).
import os
os.makedirs('engines', exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk
%cd /content/reels/engines/MuseTalk
# install MuseTalk's deps EXCEPT torch/mm* (we pin those ourselves to versions with wheels)
!grep -viE '^(torch|torchvision|torchaudio|mmcv|mmdet|mmpose|mmengine)' requirements.txt > /tmp/req.txt || cp requirements.txt /tmp/req.txt
!pip -q install -r /tmp/req.txt
!pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
!pip -q install -U openmim
!mim install mmengine
!mim install 'mmcv==2.1.0'
!mim install 'mmdet==3.1.0'
!mim install 'mmpose==1.1.0'
# static ffmpeg for MuseTalk
if not os.path.isdir('ffmpeg') or not os.path.exists('ffmpeg/ffmpeg'):
    !mkdir -p ffmpeg && cd ffmpeg && wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz --strip-components=1
os.environ['FFMPEG_PATH'] = '/content/reels/engines/MuseTalk/ffmpeg'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
!python -c "from mmpose.apis import inference_topdown; print('mmpose import OK')"

In [ ]:
# 6) Download MuseTalk weights (~5 GB, one-time)
%cd /content/reels/engines/MuseTalk
!sh ./download_weights.sh || bash ./download_weights.sh

In [ ]:
# 7) Your script -> voice (edge-tts). Edit the script or upload your own photo.
%cd /content/reels
SCRIPT = 'examples/script.txt'          # or upload your own .txt
AVATAR = 'avatars/panditji.png'         # or upload your own front-facing photo
VOICE  = 'hi-IN-MadhurNeural'           # hi-IN-SwaraNeural (F), en-IN-PrabhatNeural ...
!python tts.py --script "$SCRIPT" --engine edge --voice "$VOICE" --output output/speech.wav
from IPython.display import Audio; Audio('output/speech.wav')

In [ ]:
# 8) Run MuseTalk lip-sync (image + audio -> talking video)
import yaml, os
os.chdir('/content/reels/engines/MuseTalk')
cfg = {'task_0': {'video_path': '/content/reels/' + AVATAR, 'audio_path': '/content/reels/output/speech.wav'}}
os.makedirs('configs/inference', exist_ok=True)
open('configs/inference/reels.yaml', 'w').write(yaml.dump(cfg))
!python -m scripts.inference \
  --inference_config configs/inference/reels.yaml \
  --result_dir /content/reels/output/muse \
  --version v15 --fps 25 --batch_size 4 --use_float16 \
  --parsing_mode jaw --extra_margin 10

In [ ]:
# 9) Compose a 9:16 reel + preview/download
%cd /content/reels
import glob, os
clips = sorted(glob.glob('output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
assert clips, 'No MuseTalk output found — check the previous cell logs.'
raw = clips[-1]; print('raw clip:', raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
from google.colab import files
from IPython.display import Video
files.download('output/reel.mp4')
Video('output/reel.mp4', embed=True, width=270)